In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('csv/train.csv')

for col in df.select_dtypes(include=['object', 'string', 'str']).columns:
    df[col] = df[col].str.strip()

df.replace('NaN', np.nan)
df.dropna(subset=['Time_Orderd', 'Time_Order_picked', 'Time_taken(min)'], inplace=True)

df['Time_taken(min)'] = df['Time_taken(min)'].str.extract(r'(\d+)').astype(float)

df['Order_Date'] = pd.to_datetime(df['Order_Date'], format="%d-%m-%Y")

df['Time_Orderd'] = df['Time_Orderd'].apply(lambda x: str(x) + ':00' if len(str(x)) <= 5 else str(x))
df['Time_Order_picked'] = df['Time_Order_picked'].apply(lambda x: str(x) + ':00' if len(str(x)) <= 5 else str(x))

df['Time_Orderd'] = pd.to_timedelta(df['Time_Orderd'], errors='coerce')
df['Time_Order_picked'] = pd.to_timedelta(df['Time_Order_picked'], errors='coerce')

df.dropna(subset=['Time_Orderd', 'Time_Order_picked'], inplace=True)

time_diff = (df['Time_Order_picked'] - df['Time_Orderd']).dt.total_seconds() / 60

df['prep_time'] = np.where(time_diff < 0, time_diff + 1440, time_diff)
df['travel_time'] = df['Time_taken(min)'] - df['prep_time']

df = df[(df['prep_time'] > 0) & (df['prep_time'] < 60)]
df = df[(df['travel_time'] > 0) & (df['travel_time'] < 120)]

print(df[['Time_Orderd', 'Time_Order_picked', 'prep_time', 'travel_time', 'Time_taken(min)']].head())

      Time_Orderd Time_Order_picked  prep_time  travel_time  Time_taken(min)
0 0 days 11:30:00   0 days 11:45:00       15.0          9.0             24.0
1 0 days 19:45:00   0 days 19:50:00        5.0         28.0             33.0
2 0 days 08:30:00   0 days 08:45:00       15.0         11.0             26.0
3 0 days 18:00:00   0 days 18:10:00       10.0         11.0             21.0
4 0 days 13:30:00   0 days 13:45:00       15.0         15.0             30.0


In [4]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    
    a = np.sin(delta_phi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return R * c

df['distance_km'] = haversine(
    df['Restaurant_latitude'], df['Restaurant_longitude'],
    df['Delivery_location_latitude'], df['Delivery_location_longitude']
)

df = df[(df['distance_km'] > 0.5) & (df['distance_km'] < 25.0)]

categorical_columns = ['Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle', 'Festival', 'City']
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

print("Spatial Math and Encoding Complete!")
print(f"Final shape of the dataset ready for the Neural Network: {df.shape}")

Spatial Math and Encoding Complete!
Final shape of the dataset ready for the Neural Network: (41661, 33)


In [8]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

cols_to_drop = ['ID', 'Delivery_person_ID', 'Order_Date', 'Time_Orderd', 
                'Time_Order_picked', 'Time_taken(min)', 'prep_time', 'travel_time']

X = df.drop(columns=cols_to_drop, errors='ignore')

X = X.select_dtypes(include=['number'])

X = X.fillna(X.mean())

y_prep = df['prep_time']
y_travel = df['travel_time']

X_train, X_test, y_prep_train, y_prep_test, y_travel_train, y_travel_test = train_test_split(
    X, y_prep, y_travel, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [9]:
input_layer = Input(shape=(X_train_scaled.shape[1],), name='input_features')

shared_1 = Dense(64, activation='relu')(input_layer)
shared_2 = Dense(32, activation='relu')(shared_1)

prep_dense = Dense(16, activation='relu')(shared_2)
prep_output = Dense(1, name='prep_output')(prep_dense)

travel_dense = Dense(16, activation='relu')(shared_2)
travel_output = Dense(1, name='travel_output')(travel_dense)

model = Model(inputs=input_layer, outputs=[prep_output, travel_output])

model.compile(optimizer='adam', 
              loss={'prep_output': 'mse', 'travel_output': 'mse'},
              loss_weights={'prep_output': 0.4, 'travel_output': 0.6})

model.summary()

E0000 00:00:1779041035.726493  128158 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_features      │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        448 │ input_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16)        │        528 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 16)        │        528 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prep_output (Dense) │ (None, 1)         │         17 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ travel_output       │ (None, 1)         │         17 │ dense_3[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,618 (14.13 KB)

 Trainable params: 3,618 (14.13 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
print("Starting Training...\n")
history = model.fit(
    X_train_scaled, 
    [y_prep_train, y_travel_train], 
    validation_data=(X_test_scaled, [y_prep_test, y_travel_test]),
    epochs=20, 
    batch_size=32
)
model.save('mimo_model.keras')
print("\nModel saved successfully as 'mimo_model.keras'")

Starting Training...

Epoch 1/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 54.2615 - prep_output_loss: 16.5573 - travel_output_loss: 79.3837 - val_loss: 54.8423 - val_prep_output_loss: 16.5401 - val_travel_output_loss: 80.4200
Epoch 2/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 54.2358 - prep_output_loss: 16.5728 - travel_output_loss: 79.3443 - val_loss: 54.9342 - val_prep_output_loss: 16.5669 - val_travel_output_loss: 80.5482
Epoch 3/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 54.1454 - prep_output_loss: 16.5257 - travel_output_loss: 79.2164 - val_loss: 54.7843 - val_prep_output_loss: 16.4915 - val_travel_output_loss: 80.3536
Epoch 4/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 54.0973 - prep_output_loss: 16.4954 - travel_output_loss: 79.1500 - val_loss: 54.8769 - val_prep_output_loss: 16.5343 - val_travel_output_loss: 80.4762
Epoch 5/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 54.0118 - prep_output_loss: 16.5213 - travel_output_loss: 79